# TLIO Car-IMU Simulation

**Goal:** Adapt the trained TLIO v2.5 displacement network for a **ground-vehicle (car)** scenario.

Cars move on a planar surface (X-Y) with negligible vertical (Z) motion. We:
1. Load the trained `GHOST_v25_TFLM.keras` model — **no retraining**.
2. Filter `tlio_golden` for windows with planar ground-truth motion (|Δz| < 5cm).
3. Evaluate per-axis MAE on this car-like subset.
4. Visualize predictions: top-down (X-Y) displacement panels + cumulative trajectory drift.
5. Produce figures for the scientific paper draft (arXiv submission).

All figures are saved to `paper/figures/*.png` at 300 DPI.

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

from load_data_tf import load_tlio_sequence, create_windows, WINDOW_SIZE, STRIDE
from train_tf import nll_loss, displacement_mae

# Output directory for paper figures
FIG_DIR = os.path.join('paper', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

# Style settings — publication-quality
plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

print(f'TensorFlow: {tf.__version__}')
print(f'Output figures dir: {FIG_DIR}')

## 1. Load Pre-Trained TLIO v2.5 Model

Load weights trained on the full 3D dataset. We do **not** retrain — we evaluate the existing model on a planar subset.

In [ ]:
MODEL_PATH = 'GHOST_v25_TFLM.keras'

model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={'nll_loss': nll_loss, 'displacement_mae': displacement_mae},
    compile=False,
)
model.compile(
    optimizer='adam',
    loss=nll_loss,
    metrics=[displacement_mae],
)

print(f'Loaded model: {model.count_params():,} parameters')
print(f'Input shape:  {model.input_shape}')
print(f'Output shape: {model.output_shape}')

## 2. Load tlio_golden Sequences

We use the same sequence-loading utility as `lab_tensor.ipynb`. Update `TLIO_BASE` if needed.

In [ ]:
TLIO_BASE = r'E:\EKF2\golden-new-format-cc-by-nc-with-imus-v1.5\tlio_golden'

all_seq_ids = []
for list_file in ['train_list.txt', 'val_list.txt', 'test_list.txt']:
    p = os.path.join(TLIO_BASE, list_file)
    if os.path.exists(p):
        with open(p) as f:
            all_seq_ids += [line.strip() for line in f if line.strip()]

print(f'Total tlio_golden sequences: {len(all_seq_ids)}')

# Use first 100 sequences for the car-evaluation. Adjust if memory tight.
EVAL_SEQ_LIMIT = 100
sequences_to_load = all_seq_ids[:EVAL_SEQ_LIMIT]

all_X, all_y, seq_lengths = [], [], []
failed = []
for i, seq_id in enumerate(sequences_to_load):
    try:
        imu, pos, quat = load_tlio_sequence(os.path.join(TLIO_BASE, seq_id))
        X, y = create_windows(imu, pos, quat, WINDOW_SIZE, STRIDE)
        all_X.append(X)
        all_y.append(y)
        seq_lengths.append(len(X))
        if (i + 1) % 25 == 0:
            print(f'  Loaded {i+1}/{len(sequences_to_load)}')
    except Exception as e:
        failed.append(seq_id)

X_all = np.concatenate(all_X)
y_all = np.concatenate(all_y)
print(f'Total windows: {X_all.shape[0]}, X shape: {X_all.shape}')
if failed:
    print(f'Skipped {len(failed)} sequences with load errors.')

## 3. Filter for Car-Like (Planar) Motion

**Definition.** A *car-like window* is one where the body-frame ground-truth Z-displacement is small:
$$|\Delta z_{body}| < 0.05\ \text{m}$$

Cars move on roads — vertical displacement over 1 second is typically dominated by suspension noise (≪ 5 cm). Walking and drone data with significant vertical motion are excluded.

In [ ]:
Z_THRESHOLD = 0.05  # 5 cm vertical displacement threshold

planar_mask = np.abs(y_all[:, 2]) < Z_THRESHOLD
X_car = X_all[planar_mask]
y_car = y_all[planar_mask]

keep_pct = 100.0 * planar_mask.sum() / len(planar_mask)
print(f'Total windows:    {len(planar_mask)}')
print(f'Planar windows:   {planar_mask.sum()}  ({keep_pct:.1f}% kept)')
print(f'Removed windows:  {(~planar_mask).sum()}  ({100.0-keep_pct:.1f}% with |Δz| ≥ {Z_THRESHOLD}m)')

# Verify the filter worked
print(f'\nGround-truth Z-displacement statistics on car subset:')
print(f'  mean |Δz| = {np.mean(np.abs(y_car[:, 2])):.4f} m')
print(f'  max  |Δz| = {np.max(np.abs(y_car[:, 2])):.4f} m')

## 4. Run Inference on Car Subset

In [ ]:
raw_pred = model.predict(X_car, batch_size=128, verbose=1)
pred_disp = raw_pred[:, :3]
pred_logvar = raw_pred[:, 3:]
pred_std = np.exp(pred_logvar / 2.0)

errors_3d = np.linalg.norm(pred_disp - y_car, axis=1)
errors_2d = np.linalg.norm(pred_disp[:, :2] - y_car[:, :2], axis=1)
per_axis_mae = np.mean(np.abs(pred_disp - y_car), axis=0)
avg_std = np.mean(pred_std, axis=0)

print('=' * 60)
print('Car-IMU Evaluation Results')
print('=' * 60)
print(f'  Samples:          {len(errors_3d):,}')
print(f'  Mean 3D error:    {errors_3d.mean():.4f} m')
print(f'  Mean 2D error:    {errors_2d.mean():.4f} m  (X-Y plane only)')
print(f'  Median 3D error:  {np.median(errors_3d):.4f} m')
print(f'  Per-axis MAE:     dx={per_axis_mae[0]:.4f}, dy={per_axis_mae[1]:.4f}, dz={per_axis_mae[2]:.4f} m')
print(f'  Avg uncertainty:  σx={avg_std[0]:.4f}, σy={avg_std[1]:.4f}, σz={avg_std[2]:.4f} m')
print()
print('  Reference (full 3D dataset, before car-filtering): mean 3D error = 0.1483 m')
print('  Reference (v1 baseline ResNet-1D):                  mean 3D error = 0.4029 m')

## 5. Figure 1 — Top-Down (X-Y) Displacement Panel

12 representative windows. Each panel shows:
- **Green arrow**: ground-truth body-frame displacement
- **Red arrow**: TLIO predicted displacement
- **Red ellipse**: 1-σ uncertainty (predicted log-variance for X, Y)
- Origin marker, dashed orange error line

In [ ]:
NUM_SAMPLES = 12
cols = 4
rows = (NUM_SAMPLES + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, 4.0 * rows), facecolor='white')
axes = axes.flatten()

indices = np.linspace(0, len(pred_disp) - 1, NUM_SAMPLES, dtype=int)

for i, idx in enumerate(indices):
    ax = axes[i]
    gt = y_car[idx]
    pr = pred_disp[idx]
    sx, sy = pred_std[idx, 0], pred_std[idx, 1]

    # Origin
    ax.plot(0, 0, marker='o', color='blue', markersize=8, zorder=5, label='Origin' if i == 0 else None)
    # Ground truth (green)
    ax.annotate('', xy=(gt[0], gt[1]), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='green', lw=2),
                annotation_clip=False)
    # Prediction (red)
    ax.annotate('', xy=(pr[0], pr[1]), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='red', lw=2),
                annotation_clip=False)
    # Error line (orange dashed)
    ax.plot([gt[0], pr[0]], [gt[1], pr[1]], 'orange', linestyle='--', lw=1.0, alpha=0.7)
    # Uncertainty ellipse (1-sigma)
    ell = Ellipse(xy=(pr[0], pr[1]), width=2*sx, height=2*sy,
                  edgecolor='red', facecolor='red', alpha=0.10, lw=0.6)
    ax.add_patch(ell)

    err = errors_2d[idx]
    ax.set_title(f'#{idx}: 2D err = {err:.3f} m', fontsize=10)
    ax.set_xlabel('Δx (forward, m)', fontsize=8)
    ax.set_ylabel('Δy (right, m)', fontsize=8)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    rmax = max(np.abs(np.concatenate([gt[:2], pr[:2]])).max(), 0.1)
    ax.set_xlim(-rmax * 1.3, rmax * 1.3)
    ax.set_ylim(-rmax * 1.3, rmax * 1.3)

# Hide unused
for j in range(NUM_SAMPLES, len(axes)):
    axes[j].set_visible(False)

# Custom legend
from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=8, label='Origin'),
    Line2D([0], [0], color='green', lw=2, label='Ground truth'),
    Line2D([0], [0], color='red',   lw=2, label='TLIO prediction'),
    Line2D([0], [0], color='orange', lw=1, linestyle='--', label='Error'),
]
fig.legend(handles=legend_handles, loc='upper center', ncol=4, frameon=False, fontsize=10)
fig.suptitle('Car-IMU TLIO: Top-Down (X-Y) Predicted vs Ground-Truth Displacement',
             fontsize=13, fontweight='bold', y=1.00)
plt.tight_layout(rect=[0, 0, 1, 0.97])
out_path = os.path.join(FIG_DIR, 'displacement_xy_panel.png')
plt.savefig(out_path)
print(f'Saved: {out_path}')
plt.show()

## 6. Figure 2 — Trajectory Accumulation

Pick one tlio_golden sequence, treat consecutive predictions as displacement deltas, and accumulate them into a 2-D trajectory. Compare with ground truth path. This shows **drift growth over time** — the metric that ultimately matters for navigation.

In [ ]:
# Find a 'long' sequence to make the trajectory plot visually meaningful
if len(seq_lengths) == 0:
    raise RuntimeError('No sequences loaded.')

best_seq = int(np.argmax(seq_lengths))
start_idx = sum(seq_lengths[:best_seq])
end_idx = start_idx + seq_lengths[best_seq]
print(f'Using sequence #{best_seq} ({seq_lengths[best_seq]} windows, indices {start_idx}-{end_idx})')

X_seq = X_all[start_idx:end_idx]
y_seq = y_all[start_idx:end_idx]
raw_seq = model.predict(X_seq, batch_size=128, verbose=0)
pred_seq = raw_seq[:, :3]

# Project both to X-Y plane (zero out Z to enforce car constraint)
y_seq_xy   = np.copy(y_seq);   y_seq_xy[:, 2]   = 0.0
pred_seq_xy = np.copy(pred_seq); pred_seq_xy[:, 2] = 0.0

# Cumulative trajectories
traj_gt   = np.cumsum(y_seq_xy,   axis=0)
traj_pred = np.cumsum(pred_seq_xy, axis=0)

# Drift = euclidean distance over time
drift = np.linalg.norm(traj_pred - traj_gt, axis=1)
time_s = np.arange(len(drift)) * (STRIDE / 200.0)  # window stride / IMU rate

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), facecolor='white')

ax1.plot(traj_gt[:, 0],   traj_gt[:, 1],   'g-', lw=2.0, label='Ground truth path')
ax1.plot(traj_pred[:, 0], traj_pred[:, 1], 'r-', lw=1.5, alpha=0.85, label='TLIO accumulated')
ax1.plot(0, 0, marker='o', color='blue', markersize=10, zorder=5, label='Start')
ax1.plot(traj_gt[-1, 0], traj_gt[-1, 1], marker='*', color='green', markersize=12, zorder=5)
ax1.plot(traj_pred[-1, 0], traj_pred[-1, 1], marker='*', color='red', markersize=12, zorder=5)
ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.set_title(f'Top-Down Trajectory ({len(drift)} windows = {time_s[-1]:.1f}s)')
ax1.set_aspect('equal')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

ax2.plot(time_s, drift, 'r-', lw=2)
ax2.fill_between(time_s, 0, drift, alpha=0.2, color='red')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Position drift (m)')
ax2.set_title('Cumulative Position Drift Over Time')
ax2.grid(True, alpha=0.3)

fig.suptitle('Car-IMU Trajectory Accumulation', fontsize=13, fontweight='bold')
plt.tight_layout()
out_path = os.path.join(FIG_DIR, 'trajectory.png')
plt.savefig(out_path)
print(f'Saved: {out_path}')
print(f'Final drift after {time_s[-1]:.1f}s: {drift[-1]:.2f} m')
plt.show()

## 7. Figure 3 — Per-Axis Error Distributions

Histogram of prediction errors per axis (dx, dy, dz). For car motion we expect:
- **dx, dy** errors centered near zero with similar spread (planar motion)
- **dz** errors very small (≪ 5 cm) since ground truth Z is constrained

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), facecolor='white')
axis_names = ['Δx (forward)', 'Δy (right)', 'Δz (vertical)']
axis_colors = ['#2a8aff', '#ff8a2a', '#9e2aff']

for k, (name, color) in enumerate(zip(axis_names, axis_colors)):
    err = pred_disp[:, k] - y_car[:, k]
    axes[k].hist(err, bins=80, color=color, edgecolor='black', alpha=0.85)
    axes[k].axvline(0, color='black', linestyle='--', lw=1)
    axes[k].axvline(err.mean(), color='red', linestyle='-', lw=1.5,
                    label=f'mean = {err.mean():.4f} m')
    axes[k].set_title(f'{name}  |  σ = {err.std():.4f} m')
    axes[k].set_xlabel('Prediction error (m)')
    axes[k].set_ylabel('Count')
    axes[k].legend(loc='upper right')
    axes[k].grid(True, alpha=0.3)

fig.suptitle('Car-IMU Prediction Error Distribution per Axis', fontsize=13, fontweight='bold')
plt.tight_layout()
out_path = os.path.join(FIG_DIR, 'per_axis_error.png')
plt.savefig(out_path)
print(f'Saved: {out_path}')
plt.show()

## 8. Save Quantitative Results to a CSV (for paper tables)

In [ ]:
import csv

out_csv = os.path.join('paper', 'results_summary.csv')
with open(out_csv, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['metric', 'value_meters'])
    w.writerow(['samples', len(errors_3d)])
    w.writerow(['mean_3d_error', f'{errors_3d.mean():.4f}'])
    w.writerow(['mean_2d_error', f'{errors_2d.mean():.4f}'])
    w.writerow(['median_3d_error', f'{np.median(errors_3d):.4f}'])
    w.writerow(['mae_dx', f'{per_axis_mae[0]:.4f}'])
    w.writerow(['mae_dy', f'{per_axis_mae[1]:.4f}'])
    w.writerow(['mae_dz', f'{per_axis_mae[2]:.4f}'])
    w.writerow(['avg_sigma_x', f'{avg_std[0]:.4f}'])
    w.writerow(['avg_sigma_y', f'{avg_std[1]:.4f}'])
    w.writerow(['avg_sigma_z', f'{avg_std[2]:.4f}'])
    w.writerow(['final_drift', f'{drift[-1]:.2f}'])
    w.writerow(['drift_duration_s', f'{time_s[-1]:.1f}'])
    w.writerow(['planar_window_keep_pct', f'{keep_pct:.1f}'])

print(f'Saved: {out_csv}')
print('\nUse these numbers to populate the Results table in main.tex.')

## Done

Outputs ready for the paper:

- `paper/figures/displacement_xy_panel.png` — Figure 1 (top-down displacement panel)
- `paper/figures/trajectory.png` — Figure 2 (cumulative trajectory + drift)
- `paper/figures/per_axis_error.png` — Figure 3 (per-axis error histograms)
- `paper/results_summary.csv` — quantitative results for paper tables

Next: compile `paper/main.tex` to PDF and bundle for arXiv.